# Scraping des publications d'OpenAlex

Date : novembre 2025 (la base de données a pu être mise à jour depuis)

In [1]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from collections import Counter
from itertools import chain
import math


In [2]:
BASE_URL = "https://api.openalex.org/works"

params = {
    "search": '"Vietnam" OR "Viêtnam" OR "Viet Nam" OR "Viêt Nam" OR "Indochina" OR "Indochine"',
    "filter": "from_publication_date:1975-01-01,to_publication_date:2024-12-31,authorships.institutions.country_code:FR|US|VN",
    "per_page": 200,                      
    "cursor": "*",                              
}

all_works = []
n_pages_max = 659  # après avoir regardé le json

# boucle principale sur les pages
for i in range(n_pages_max):
    print(f"Page {i+1}")
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    data = r.json()

    results = data.get("results", [])
    if not results:
        break

    all_works.extend(results)
    next_cursor = data.get("meta", {}).get("next_cursor")
    if not next_cursor:
        break
    params["cursor"] = next_cursor

    time.sleep(0.2)

print(f"Nombre total de publications récupéréés : {len(all_works)}")

Page 1


KeyboardInterrupt: 

Fonctions pour récupérer des informations

In [ ]:
def get_abstract_text(work):
    """Reconstruit l'abstract lisible à partir de abstract_inverted_index."""
    idx = work.get("abstract_inverted_index")
    if not idx:
        return None

    max_pos = 0
    for positions in idx.values():
        if positions:
            max_pos = max(max_pos, max(positions))

    words = [""] * (max_pos + 1)
    for word, positions in idx.items():
        for p in positions:
            if 0 <= p < len(words):
                words[p] = word

    text = " ".join(words).strip()
    return text or None


def extract_institution_countries(work):
    """Liste des codes pays de toutes les institutions (FR, US, etc.)."""
    countries = set()
    for auth in work.get("authorships", []):
        for inst in auth.get("institutions", []):
            code = inst.get("country_code")
            if code:
                countries.add(code)
    return "; ".join(sorted(countries)) if countries else None


def extract_institution_names(work):
    """Liste des noms d’institutions, toutes affiliations confondues."""
    inst_names = set()
    for auth in work.get("authorships", []):
        for inst in auth.get("institutions", []):
            name = inst.get("display_name")
            if name:
                inst_names.add(name)
    return "; ".join(sorted(inst_names)) if inst_names else None

In [ ]:
def extract_authors(work):
    """Liste des noms des auteurs, dans l’ordre."""
    names = []
    for auth in work.get("authorships", []):
        name = auth.get("author", {}).get("display_name")
        if name:
            names.append(name)
    return "; ".join(names) if names else None


def extract_authors_with_affiliations(work):
    """
    Pour chaque auteur, concatène : Nom (Aff1 | Aff2)
    Puis sépare les auteurs par '; '
    """
    entries = []
    for auth in work.get("authorships", []):
        name = auth.get("author", {}).get("display_name")
        if not name:
            continue
        insts = [inst.get("display_name") for inst in auth.get("institutions", [])if inst.get("display_name")]
        if insts:
            entry = f"{name} ({' | '.join(insts)})"
        else:
            entry = name
        entries.append(entry)

    return "; ".join(entries) if entries else None


def extract_concepts(work):
    """Concepts (sujets) avec leur niveau et score."""
    domains, fields, subfields, topics = [], [], [], []

    for c in work.get("concepts", []):
        level = c.get("level")
        name = c.get("display_name")
        if level == 0:
            domains.append(name)
        elif level == 1:
            fields.append(name)
        elif level == 2:
            subfields.append(name)
        elif level == 3:
            topics.append(name)
    return "; ".join(domains), "; ".join(fields),"; ".join(subfields),"; ".join(topics)


def extract_primary_topic_info(work):
    """Sujet principal (primary_topic) + field + subfield (+ domain). Retourne un dict à fusionner dans la ligne du DataFrame."""
    pt = work.get("primary_topic") or {}
    subfield = pt.get("subfield") or {}
    field = pt.get("field") or {}
    domain = pt.get("domain") or {}

    return {
        "primary_topic_id": pt.get("id"),
        "primary_topic_name": pt.get("display_name"),
        "primary_topic_score": pt.get("score"),

        "primary_topic_field_id": field.get("id"),
        "primary_topic_field_name": field.get("display_name"),

        "primary_topic_subfield_id": subfield.get("id"),
        "primary_topic_subfield_name": subfield.get("display_name"),

        "primary_topic_domain_id": domain.get("id"),
        "primary_topic_domain_name": domain.get("display_name"),
    }

def extract_primary_location_source(work):
    """Infos sur le lieu principal de publication (journal, conference, repository, etc.)via primary_location.source."""
    pl = work.get("primary_location") or {}
    src = pl.get("source") or {}

    issn_list = src.get("issn") or []
    if isinstance(issn_list, list):
        issn_str = ";".join(issn_list)
    else:
        issn_str = str(issn_list) if issn_list else None

    return {"primary_source_id": src.get("id"),"primary_source_name": src.get("display_name"),"primary_source_type": src.get("type"), "primary_source_issn_l": src.get("issn_l"),"primary_source_issn": issn_str,}

def extract_references(work) : 
    """extracion des références citées"""
    refs = work.get("referenced_works", [])
    referenced_count = len(refs)
    return "; ".join(refs), referenced_count
    



Sauvegarde 

In [ ]:
# création du dataframe
def flatten_work(w):
    base = {
        "id": w.get("id"),
        "doi": w.get("doi"),
        "title": w.get("title"),
        "publication_year": w.get("publication_year"),
        "publication_date": w.get("publication_date"),
        "cited_by_count": w.get("cited_by_count"),
        "language": w.get("language"),
        "type": w.get("type"),

        "institution_countries": extract_institution_countries(w),
        "institutions": extract_institution_names(w),

        "is_oa": w.get("is_oa"),
        "oa_status": w.get("oa_status"),
        "oa_url": w.get("oa_url"),
    }
    base.update({
        "abstract": get_abstract_text(w),
        "domains": extract_concepts(w)[0],
        "fields" : extract_concepts(w)[1],
        "subfields" : extract_concepts(w)[2],
        "topics" : extract_concepts(w)[3],
    })

    # topic principal (nom, domaine, sous-domaine associés)
    base.update(extract_primary_topic_info(w))
    base.update(extract_primary_location_source(w))

    # références citées
    base["referenced_ids"] = extract_references(w)[0]
    base["referenced_count"] = extract_references(w)[1]
    return base

# sauvegarde

rows = [flatten_work(w) for w in all_works]
df = pd.DataFrame(rows)

colonnes finalement jugées inutiles ou vides

In [101]:
df = df.drop(columns=["is_oa", "oa_status", "oa_url", 'primary_topic_score', 'primary_topic_field_id','primary_topic_subfield_id','primary_topic_domain_id',"primary_source_id",'primary_source_issn_l',"primary_source_issn"])

In [ ]:
df.to_csv("openalex_vietnam_fr_us_ready_ter.csv", index = False, encoding="utf-8")


NameError: name 'df' is not defined